## Complete source audit and OTC integration

Use the original FDA archives and one complete RxNorm release. Keep prescribable files separate. Source agreement records submitted evidence and does not establish independent clinical verification. Ingredient identity and formulation confidence have separate eligibility rules.

FDA UNII identifiers can normalize aliases when the source supports them. Unordered identifier lists never assign individual ingredient strengths. OTC and prescription classification disagreement remains visible even when ingredient identity agrees.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
assert (project_root / "src/hrp").is_dir()
source_dir = Path(os.environ.get("HRP_SOURCE_DIR", str(project_root.parent / "data")))

from hrp.data_loading import verify_sources
from hrp.ingredient_agreement import run_integration

source_audit = verify_sources(source_dir, project_root / "results/final_v2/part01")
print(json.dumps({key: source_audit[key] for key in ("ndc", "label", "label_shards")}, indent=2))
display(pd.DataFrame(source_audit["rxnorm"])[["file", "rows", "wrong_column_count", "duplicate_rows"]])


Reused complete source audit after verifying every file hash


{
  "ndc": {
    "source": "ndc",
    "records": 138333,
    "unique_primary_ids": 135898,
    "missing_primary_ids": 0,
    "duplicate_primary_ids": 2435,
    "malformed_active_ingredient_lists": 0
  },
  "label": {
    "source": "label",
    "records": 264736,
    "unique_primary_ids": 264736,
    "missing_primary_ids": 0,
    "duplicate_primary_ids": 0,
    "unique_set_ids": 264736
  },
  "label_shards": {
    "shards_named": 14,
    "declared_shards": 14,
    "missing_shards": []
  }
}


,file,rows,wrong_column_count,duplicate_rows
0,RXNATOMARCHIVE.RRF,373584,0,0
1,RXNCONSO.RRF,1214380,0,0
2,RXNCUI.RRF,30269,0,0
3,RXNCUICHANGES.RRF,30,0,0
4,RXNDOC.RRF,3445,0,0
5,RXNREL.RRF,7509763,0,0
6,RXNSAB.RRF,13,0,0
7,RXNSAT.RRF,7769848,0,0
8,RXNSTY.RRF,492418,0,0


In [2]:
coverage = run_integration(project_root, source_dir)
assert sum(coverage["ndc_record_outcomes"].values()) == coverage["ndc_records"]
assert sum(coverage["exclusive_outcomes"].values()) == coverage["unique_otc_ndcs"]
assert sum(coverage["label_linkage"]["label_record_outcomes"].values()) == source_audit["label"]["records"]
print(json.dumps({key: coverage[key] for key in (
    "otc_source_records", "unique_otc_ndcs", "legacy_consistency", "ingredient_eligible",
    "formulation_eligible", "exclusive_outcomes", "invalid_package_relationships",
)}, indent=2))
display(pd.Series(coverage["ingredient_support_tiers"], name="products").to_frame())
display(pd.Series(coverage["strength_issues_nonexclusive"], name="products").to_frame())

print("FDA identity normalization", coverage["ingredient_identity_normalization"])
print("Source row outcomes", coverage["ndc_record_outcomes"])
print("Cross type listings", source_audit["ndc_product_type_relationships"]["cross_type_otc_ndcs"])

Reused complete source audit after verifying every file hash


Reused hash-verified complete label linkage


Reused hash-verified full-release RxNorm links


{
  "version": "final_v5",
  "ingredient_identity_normalization": {
    "statuses": {
      "FDA_UNII_SINGLE_INGREDIENT": 37690,
      "FDA_UNII_SET_VALIDATED": 11878,
      "FDA_UNII_SET_WITHOUT_NAME_ALIGNMENT": 4802,
      "FDA_NAME_ONLY": 1203,
      "FDA_UNII_SET_UNRESOLVED": 235,
      "FDA_UNII_NAME_CROSSWALK": 492
    },
    "ingredient_records_recovered_by_identifier_paths": 4277,
    "singleton_name_unii_links": 1460,
    "ambiguous_name_links": 0,
    "comparison_eligible": 53146,
    "rule": "Use a product singleton UNII directly. A complete product UNII set supports unordered ingredient comparison when cardinality matches and known name links do not contradict it. Unknown name alignment still withholds formulation quantities. For multiple ingredients use unique singleton-derived name mappings only when they bijectively cover the product UNII set, when present. Never pair unordered arrays by position. Equal UNII identifiers support aliases, without inferring salt, moiety, de

{
  "otc_source_records": 56982,
  "unique_otc_ndcs": 56300,
  "legacy_consistency": {
    "source_records": 56982,
    "unique_ndcs": 56300,
    "consistent_ndcs": 55250,
    "pending_ndcs": 1050,
    "consistent_source_records": 55913,
    "pending_records": 1069,
    "pending_missing_ingredients": 1039,
    "pending_missing_identity": 0,
    "pending_conflicting_fields": 11
  },
  "ingredient_eligible": 53562,
  "formulation_eligible": 22070,
  "exclusive_outcomes": {
    "accepted_ingredient": 53562,
    "rxnorm_ingredient_names_unresolved": 1700,
    "kit_requires_component_identity": 3,
    "missing_ingredient_identity": 1020,
    "fda_label_ingredient_conflict": 7,
    "conflicting_ndc_rows": 8
  },
  "invalid_package_relationships": [
    {
      "product_ndc": "53209-2000",
      "package_ndc": "02",
      "reason": "package_parent_does_not_match_product_ndc"
    }
  ]
}


,products
FDA_RXNORM_IDENTIFIER_PATH,4277
NAME_NORMALIZATION_UNRESOLVED,1700
FDA_RXNORM_AGREEMENT,22532
FDA_ONLY,26756
UNKNOWN,1020
SOURCE_CONFLICT,15


,products
label_strength_basis_unresolved,12594
complete_label_list_supplements_missing_ndc_ingredients,3
rxnorm_route_or_form_conflict,4176
ingredient_identity_not_normalized,6240
rxnorm_strength_conflict,1835
rxnorm_strength_basis_unresolved,3862
unresolved_strength_or_form,13813
fda_label_strength_conflict,97


FDA identity normalization {'statuses': {'FDA_UNII_SINGLE_INGREDIENT': 37690, 'FDA_UNII_SET_VALIDATED': 11878, 'FDA_UNII_SET_WITHOUT_NAME_ALIGNMENT': 4802, 'FDA_NAME_ONLY': 1203, 'FDA_UNII_SET_UNRESOLVED': 235, 'FDA_UNII_NAME_CROSSWALK': 492}, 'ingredient_records_recovered_by_identifier_paths': 4277, 'singleton_name_unii_links': 1460, 'ambiguous_name_links': 0, 'comparison_eligible': 53146, 'rule': 'Use a product singleton UNII directly. A complete product UNII set supports unordered ingredient comparison when cardinality matches and known name links do not contradict it. Unknown name alignment still withholds formulation quantities. For multiple ingredients use unique singleton-derived name mappings only when they bijectively cover the product UNII set, when present. Never pair unordered arrays by position. Equal UNII identifiers support aliases, without inferring salt, moiety, density or clinical equivalence. Unknown identifiers remain eligible for source-name tasks but cannot establ

Only complete short labeling lists supplement missing structured ingredients. Original strings and label quantities remain in the local evidence ledger. Percentages without a physical basis are not converted to mass or volume ratios. Active ingredient and active moiety quantities remain distinct. Records with unresolved strength can support an ingredient answer without supporting a formulation answer.
